In [1]:
# Import Essential Modules
import wfdb
import ast
import pickle
import datetime
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from keras import layers
import matplotlib.pyplot as plt
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

# config trail runs
PTBXL_PATH = "/home/student/Prathamesh's Project Pre-requisites/DataRes/physionet.org/files/ptb-xl/1.0.3/" # path to PTB-XL dataset
SAMPLING_RATE = 100 
NUM_CLASSES = 5
NUM_EPOCHS = 100
BATCH_SIZE = 32
WIDTH = 64 # default width for epoch wise double descent

2026-03-07 13:07:25.288458: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 13:07:25.307092: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-07 13:07:25.983059: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Load data
def load_ptbxl(path, sampling_rate=100):
    Y = pd.read_csv(path + 'ptbxl_database.csv', index_col='ecg_id')
    Y.scp_codes = Y.scp_codes.apply(ast.literal_eval)

    agg_df = pd.read_csv(path + 'scp_statements.csv', index_col=0)
    agg_df = agg_df[agg_df.diagnostic == 1.0]

    def aggregate_diagnostic(y_dic):
        tmp = set()
        for key in y_dic.keys():
            if key in agg_df.index:
                tmp.add(agg_df.loc[key].diagnostic_class)
        return list(tmp)

    Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

    # parse HYP_HR entries
    target_hyp_codes = {'STD', 'STE', 'HVOLT', 'LVOLT', 'VCLVH'}
    def relabel_hyp_hr(row):
        curr_class = row['diagnostic_superclass']
        scp_dict = row['scp_codes']
        if 'HYP' in curr_class:
            if any(code in scp_dict for code in target_hyp_codes):
                return ['HYP_HR']
            else:
                return ['HYP']
        return curr_class
    
    Y['diagnostic_superclass'] = Y.apply(relabel_hyp_hr, axis = 1)
    print(Y['diagnostic_superclass'].value_counts())
    
    # Keep only samples that have at least one diagnostic superclass
    Y = Y[Y['diagnostic_superclass'].map(len) >= 1]

    # Get top 6 classes
    from collections import Counter
    all_labels = [label for sublist in Y['diagnostic_superclass'] for label in sublist]
    top6 = [label for label, _ in Counter(all_labels).most_common(6)]
    print(f'Top 6 classes: {top6}')

    # Filter to samples that have at least one top-5 label
    Y['diagnostic_superclass'] = Y['diagnostic_superclass'].apply(
    lambda x: [l for l in x if l in top6]
    )
    Y = Y[Y['diagnostic_superclass'].map(len) >= 1]

    # Load signals
    filenames = Y.filename_lr if sampling_rate == 100 else Y.filename_hr
    print('Loading signals...')
    X = np.array([wfdb.rdsamp(path + f)[0] for f in filenames]) # (N, 1000, 12)

    # Split use 10 fold stratified sampling
    test_fold = 10 
    val_fold = 9

    X_test = X[np.where(Y.strat_fold == test_fold)]
    Y_test = Y[(Y.strat_fold == test_fold)].diagnostic_superclass

    X_val = X[np.where(Y.strat_fold == val_fold)]
    Y_val = Y[(Y.strat_fold == val_fold)].diagnostic_superclass

    X_train = X[np.where((Y.strat_fold != test_fold) & (Y.strat_fold != val_fold))]
    Y_train  = Y[((Y.strat_fold != test_fold) & (Y.strat_fold != val_fold))].diagnostic_superclass

    # Multi-hot encode
    mlb = MultiLabelBinarizer(classes=top6)
    mlb.fit([top6])
    y_test = mlb.transform(Y_test) # (N, 6) binary matrix
    y_val = mlb.transform(Y_val)
    y_train = mlb.transform(Y_train)

    return X_train, y_train, X_test, y_test, X_val, y_val, top6, mlb 

In [3]:
# Conv1D+BiLSTM Model

# residual block
def residual_block(x, filters, kernel_size = 3, stride=1):
    shortcut = x
    
    # main path
    x = layers.Conv1D(filters, kernel_size, strides = stride, padding = "same", use_bias = False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv1D(filters, kernel_size, strides = stride, padding = "same", use_bias = False)(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, kernel_size = 1, strides = stride, use_bias = False)(shortcut)
    shortcut = layers.BatchNormalization()(shortcut)

    # skip-connection
    x = layers.Add()([shortcut, x])
    x = layers.ReLU()(x)
    return x

def build_model(input_shape = (1000, 12), num_classes = 5, width = 64):
    # Input layers
    InputLayer = layers.Input(shape=input_shape)

    # 1D-CNN Block 1
    x = layers.Conv1D(width, kernel_size = 7, strides = 2, padding = "same", use_bias = False)(InputLayer)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    

    # residual block 1
    x = residual_block(x, width)
    # residual block 2
    x = residual_block(x, width)
    # residual block 3
    x = residual_block(x, width)

    # Bi-LSTM Block 1 (redacted for less training time)
    x = layers.Bidirectional(layers.LSTM(units = 64, return_sequences = True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(units = 32, return_sequences = False))(x)
    
    # x = layers.GlobalAveragePooling1D()(x)
    # Classification Overhead
    x = layers.Dense(units = 64, activation = "relu")(x)
    x = layers.Dropout(0.3)(x)
    OutputLayer = layers.Dense(num_classes, activation = "sigmoid")(x)

    # Model Creation
    return tf.keras.Model(inputs = InputLayer, outputs = OutputLayer, name = "1DCNN_BiLSTM_Z_Model")

In [4]:
# Define custom metrics

# subclass custom Hamming Loss metric (Not using tensorflow addons here; version clash)
@tf.keras.utils.register_keras_serializable()
class HammingLoss(tf.keras.metrics.Metric):

    def __init__(self, name = "Hamming_loss", **kwargs):
        super(HammingLoss, self).__init__(name = name, **kwargs)
        self.total_mismatches = self.add_weight(name = "Total_mismatches", initializer = 'zeros', dtype = tf.float32)
        self.total_labels = self.add_weight(name = "Total_labels", initializer = 'zeros', dtype = tf.float32)

    def update_state(self, y_true, y_pred, sample_weight=None):
        # caste predictions and targets in tf.float32
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(tf.greater(y_pred, 0.5), tf.float32)

        # Calculate mismatches
        mismatches = tf.cast((tf.math.count_nonzero(tf.math.not_equal(y_true, y_pred), axis=-1)), tf.float32)

        # Find number of labels and batch size
        num_label = tf.cast(tf.shape(y_true)[-1], tf.float32) # shape is (rows, columns) and columns = number of elements in array
        batch_size = tf.cast(tf.shape(y_true)[0], tf.float32) # shape is (rows, columns) and rows = batch size per array
        
        # Update number of mismatches and total labels count
        self.total_mismatches.assign_add(tf.reduce_sum(mismatches)) # reduce sum adds all the elements in an array (here, instance)
        self.total_labels.assign_add(batch_size * num_label) # total label count = number of labels x batch size per instance

    def result(self):
        return self.total_mismatches / self.total_labels # Hamming Loss formula
    
    def reset_state(self): # reset atttributes
        self.total_mismatches.assign(0.)
        self.total_labels.assign(0.)

In [5]:
def run_experiment(width = 64, epochs = 100, sample = 0):
    # Load data
    X_train, y_train, X_test, y_test, X_val, y_val, classes, mlb = load_ptbxl(PTBXL_PATH, SAMPLING_RATE)

    # Normalise per-lead
    X_test = (X_test-X_test.mean(axis = 1, keepdims = True)) / (X_test.std(axis = 1, keepdims = True) + 1e-8)
    X_train = (X_train-X_train.mean(axis = 1, keepdims = True)) / (X_train.std(axis = 1, keepdims = True) + 1e-8)
    X_val = (X_val-X_val.mean(axis = 1, keepdims = True)) / (X_val.std(axis = 1, keepdims = True) + 1e-8)
    
    # Build Model
    model = build_model(
    input_shape = (X_train.shape[1], X_train.shape[2]),
    num_classes = len(classes),
    width = width
    )

    # Model summary and parameters number
    model.summary()

    # Adam optimiser
    optimiser = tf.keras.optimizers.Adam(learning_rate = 1e-3)

    # Compile model
    model.compile(
    optimizer = optimiser,
    loss = tf.keras.losses.binary_focal_crossentropy,
    metrics = [tf.keras.metrics.Precision(name = "precision"), 
               tf.keras.metrics.Recall(name = "recall", thresholds = 0.30),
               tf.keras.metrics.AUC(curve = "ROC", multi_label = True),
               HammingLoss()]
    )

    # Decide callbacks
    Callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor = "val_precision", patience = 8, restore_best_weights = True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor = "val_loss", factor = 0.5, patience = 5)
    ]

    print(f"--------------------------------Training commenced--------------------------------")
    print(f"Training Set loading.....\n")

    # prep run
    history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs = epochs,
    callbacks = Callbacks
    )

    print(f"\n--------------------------------Training completed--------------------------------")
    print(f"Evaluation Set loading.....\n")
    
    # eval results
    performance = model.evaluate(
    x = X_test,
    y = y_test,
    batch_size = 32 
    )
    
    print(f"\n--------------------------------Evalution done!--------------------------------\n")
    
    # predict sample
    print(f"Showing predictions for sample number {sample}")
    prediction = model.predict(x = X_test[sample:(sample+1)])
    print(prediction)

    # Save model
    print(f"\n--------------------------------Save current model--------------------------------")
    print(f"\nSaving model....")
    model.save('optimal_hr_model.keras')

    print(f"\n--------------------------------Model saved successfully!--------------------------------")

    return history, performance, prediction

In [6]:
history, performance, prediction = run_experiment(width = 64, epochs = 100, sample = 0)

diagnostic_superclass
[NORM]              9069
[MI]                2532
[STTC]              2400
[HYP]               1966
[CD]                1708
[CD, MI]            1297
[HYP_HR]             683
[STTC, MI]           599
[STTC, CD]           471
[]                   411
[CD, NORM]           407
[STTC, CD, MI]       223
[STTC, NORM]          28
[STTC, CD, NORM]       5
Name: count, dtype: int64
Top 6 classes: ['NORM', 'MI', 'CD', 'STTC', 'HYP', 'HYP_HR']
Loading signals...


Model: "1DCNN_BiLSTM_Z_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1000, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 500, 64)   │      5,376 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 500, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 500, 64)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 500, 64)   │     12,288 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 500, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 500, 64)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 500, 64)   │     12,288 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 500, 64)   │        256 │ re_lu[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 500, 64)   │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 500, 64)   │          0 │ batch_normalizat… │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 500, 64)   │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 500, 64)   │     12,288 │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 500, 64)   │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 500, 64)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 500, 64)   │     12,288 │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 500, 64)   │        256 │ re_lu_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 500, 64)   │        256 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 500, 64)   │          0 │ batch_normalizat… │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 500, 64)   │          0 │ add_1[0][0]     

 Total params: 193,478 (755.77 KB)

 Trainable params: 192,198 (750.77 KB)

 Non-trainable params: 1,280 (5.00 KB)

--------------------------------Training commenced--------------------------------
Training Set loading.....

Epoch 1/100


2026-03-07 13:08:18.595098: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


534/534 ━━━━━━━━━━━━━━━━━━━━ 88s 159ms/step - Hamming_loss: 0.1664 - auc: 0.6714 - loss: 0.1046 - precision: 0.6278 - recall: 0.8958 - val_Hamming_loss: 0.1642 - val_auc: 0.6962 - val_loss: 0.1029 - val_precision: 0.6297 - val_recall: 0.8979 - learning_rate: 0.0010
Epoch 2/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 86s 161ms/step - Hamming_loss: 0.1465 - auc: 0.7395 - loss: 0.0939 - precision: 0.7034 - recall: 0.8936 - val_Hamming_loss: 0.1480 - val_auc: 0.7463 - val_loss: 0.0924 - val_precision: 0.7023 - val_recall: 0.8684 - learning_rate: 0.0010
Epoch 3/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 86s 161ms/step - Hamming_loss: 0.1409 - auc: 0.7710 - loss: 0.0888 - precision: 0.7239 - recall: 0.8995 - val_Hamming_loss: 0.1342 - val_auc: 0.7828 - val_loss: 0.0842 - val_precision: 0.7293 - val_recall: 0.8862 - learning_rate: 0.0010
Epoch 4/100
534/534 ━━━━━━━━━━━━━━━━━━━━ 86s 161ms/step - Hamming_loss: 0.1364 - auc: 0.7959 - loss: 0.0844 - precision: 0.7287 - recall: 0.9080 - val_Hamming_loss: 0.1321 - val_

- for patient with ecg_id 0, these were the following results:

    - ~73% probability of being a **Normal individual**.
    - ~14% probablistic risk of suffering with Myocardial Infraction.
    - ~23% probablistic risk of suffering with Cardiac Dysfunction.
    - ~15% probablistic risk of suffering with ST/T wave changes (possible cause of multilple conditions).
    - ~13% probablistic risk of suffering with Hypertrophic cardiomyopathy.
    - ~15% probablistic risk of suffering with HCM and untimely cardiac arrest; worse even.